# 02 - Feature Selection (Filtros + Modelo)

Proyecto: **Forecasting del MERVAL**  
Autor: *Santi*  
Generado: *2025-11-04 22:36*

> Este cuaderno fue autogenerado a partir de `init.ipynb` y está pensado para mantener un **estilo y estructura similares**.
> Ajustá la celda de **CONFIG** si tu dataset o columnas cambian.


## CONFIG

- Ruta dataset detectada: `../inputs/dataset_v2.csv`  
- Features pre-seleccionadas detectadas:
_No se detectaron automáticamente. Definilas abajo._

> Si la lista está vacía o querés modificarla, editá la celda de código de CONFIG a continuación.


In [ ]:
# ==== CONFIGURACIÓN GENERAL ====
import os
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# Ruta del dataset (ajustá si es necesario):
DATA_PATH = r"""../inputs/dataset_v2.csv"""

# Columna de fecha (si no existe, dejá None y usaremos el índice)
DATE_COL = "date"  # o None

# Variable objetivo y horizonte de forecast (ejemplo: retorno del MERVAL t+1)
TARGET_COL = "target"        # cambialo por tu target real
H = 1                         # horizonte en pasos

# Lista de features pre-seleccionadas (editar si es necesario)
SELECTED_FEATURES = [
    # 'col1', 'col2', 'col3'
]

# Opcional: columnas a excluir (IDs, texto libre, etc.)
EXCLUDE_COLS = []

pd.options.display.max_columns = 120
pd.options.display.width = 160


In [ ]:
# ==== CARGA DE DATOS ====
# Intentamos leer parquet/csv en orden. Ajustá DATA_PATH según tu proyecto.
df = None
if os.path.exists(DATA_PATH):
    if DATA_PATH.lower().endswith(".parquet"):
        df = pd.read_parquet(DATA_PATH)
    elif DATA_PATH.lower().endswith(".csv"):
        df = pd.read_csv(DATA_PATH)
    else:
        try:
            df = pd.read_parquet(DATA_PATH)
        except Exception:
            df = pd.read_csv(DATA_PATH)
else:
    print(f"[ADVERTENCIA] No se encontró DATA_PATH: {DATA_PATH}")

# Si hay columna de fecha, la usamos como índice temporal
if df is not None and DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.sort_values(DATE_COL).set_index(DATE_COL)

print("Forma del df:", None if df is None else df.shape)
print("Columnas:", None if df is None else list(df.columns)[:10], "...")


## Utilidades

In [ ]:
# ==== UTILIDADES ====
import numpy as np
import pandas as pd

def low_variance_filter(df, threshold: float = 1e-8):
    var = df.var(numeric_only=True)
    keep = var[var > threshold].index.tolist()
    return df[keep], keep, var

def missing_filter(df, max_na_ratio: float = 0.2):
    na_ratio = df.isna().mean()
    keep = na_ratio[na_ratio <= max_na_ratio].index.tolist()
    return df[keep], keep, na_ratio

def high_correlation_filter(df, thr: float = 0.95):
    c = df.corr(numeric_only=True).abs()
    upper = c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    drop = [col for col in upper.columns if any(upper[col] > thr)]
    keep = [c for c in df.columns if c not in drop]
    return df[keep], keep, drop

def to_supervised(df, target_col: str, horizon: int = 1):
    y = df[target_col].shift(-horizon).rename(f"{target_col}_t_plus_{horizon}")
    X = df.drop(columns=[target_col])
    out = pd.concat([X, y], axis=1).dropna(how="any")
    return out.drop(columns=[y.name]), out[y.name]


## 1) Conjunto Base para Selección

In [ ]:
# ==== BASE ====
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    fe_path_pq = "/mnt/data/features_engineered.parquet"
    fe_path_csv = "/mnt/data/features_engineered.csv"
    FE_ALL = None
    if os.path.exists(fe_path_pq):
        FE_ALL = pd.read_parquet(fe_path_pq)
    elif os.path.exists(fe_path_csv):
        FE_ALL = pd.read_csv(fe_path_csv, index_col=0, parse_dates=True)

    if FE_ALL is not None and not FE_ALL.empty:
        Xfull = FE_ALL.drop(columns=[c for c in FE_ALL.columns if c == f"{TARGET_COL}_t_plus_{H}"], errors="ignore")
        if TARGET_COL in df.columns and f"{TARGET_COL}_t_plus_{H}" in FE_ALL.columns:
            y = FE_ALL[f"{TARGET_COL}_t_plus_{H}"]
        else:
            Xfull = df[base_cols]
            y = df[TARGET_COL].shift(-H)
    else:
        Xfull = df[base_cols].copy()
        y = df[TARGET_COL].shift(-H)

    Xy = pd.concat([Xfull, y.rename("y")], axis=1).dropna(how="any")
    X = Xy.drop(columns=["y"])
    y = Xy["y"]
    print("X shape:", X.shape, "y shape:", y.shape)


## 2) Filtros por missings y low-variance

In [ ]:
# ==== MISSING + LOW VARIANCE ====
if 'X' in globals():
    X1, keep_missing, na_ratio = missing_filter(X, max_na_ratio=0.2)
    print(f"Cols tras missing filter: {len(keep_missing)} / {X.shape[1]}")
    X2, keep_lv, var = low_variance_filter(X1, threshold=1e-8)
    print(f"Cols tras low-variance: {len(keep_lv)} / {X1.shape[1]}")
    Xflt = X2.copy()
else:
    print("[INFO] Saltando: falta X")


## 3) Correlación y Mutual Information

In [ ]:
# ==== CORRELACIÓN + MI ====
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

if 'Xflt' in globals():
    X3, keep_corr, dropped_corr = high_correlation_filter(Xflt, thr=0.95)
    print(f"Cols tras correlation filter: {len(keep_corr)} / {Xflt.shape[1]} (drop {len(dropped_corr)})")

    scaler = StandardScaler(with_mean=True, with_std=True)
    Xmi = pd.DataFrame(scaler.fit_transform(X3), columns=X3.columns, index=X3.index)
    mi = mutual_info_regression(Xmi, y, random_state=SEED)
    mi_series = pd.Series(mi, index=X3.columns).sort_values(ascending=False)
    TOP_K = min(200, len(mi_series))
    top_mi_cols = mi_series.head(TOP_K).index.tolist()
    Xmi_top = X3[top_mi_cols]
    print("Top-MI cols:", len(top_mi_cols))
else:
    print("[INFO] Saltando: falta Xflt")


## 4) Selectores basados en modelo (Boruta/SHAP alternativos)

In [ ]:
# ==== MODEL-BASED ====
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

from sklearn.ensemble import RandomForestRegressor

if 'Xmi_top' in globals():
    Xm = Xmi_top.copy()
    ym = y.loc[Xm.index]

    tscv = TimeSeriesSplit(n_splits=5)
    oof_pred = pd.Series(index=Xm.index, dtype=float)
    importances = pd.Series(0.0, index=Xm.columns)

    for fold, (tr, va) in enumerate(tscv.split(Xm), 1):
        Xtr, Xva = Xm.iloc[tr], Xm.iloc[va]
        ytr, yva = ym.iloc[tr], ym.iloc[va]

        if HAS_LGB:
            model = lgb.LGBMRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=-1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=SEED,
            )
        else:
            model = RandomForestRegressor(
                n_estimators=400,
                max_depth=None,
                random_state=SEED,
                n_jobs=-1,
            )

        model.fit(Xtr, ytr)
        pred = pd.Series(model.predict(Xva), index=Xva.index)
        oof_pred.loc[Xva.index] = pred.values

        pi = permutation_importance(model, Xva, yva, n_repeats=10, random_state=SEED)
        importances = importances.add(pd.Series(pi.importances_mean, index=Xm.columns), fill_value=0.0)

    mae = mean_absolute_error(ym.loc[oof_pred.index], oof_pred)
    print(f"OOF MAE: {mae:.6f} (modelo {'LightGBM' if HAS_LGB else 'RandomForest'})")

    importances = (importances / importances.max()).fillna(0.0).sort_values(ascending=False)
    TOP_M = min(150, len(importances))
    top_model_cols = importances.head(TOP_M).index.tolist()

    Xsel = Xm[top_model_cols]
    print("Cols seleccionadas por modelo:", len(top_model_cols))

    out_cols_path = "/mnt/data/selected_features.json"
    out_rank_path = "/mnt/data/feature_importance_rank.csv"
    with open(out_cols_path, "w", encoding="utf-8") as f:
        json.dump(top_model_cols, f, ensure_ascii=False, indent=2)
    importances.to_csv(out_rank_path, header=["norm_perm_importance"])
    print(f"Guardado listado de columnas: {out_cols_path}")
    print(f"Guardado ranking importancias: {out_rank_path}")
else:
    print("[INFO] Saltando: falta Xmi_top")


### (Opcional) BorutaPy / SHAP

In [ ]:
# ==== OPCIONAL: BORUTA / SHAP ====
try:
    from boruta import BorutaPy  # pip install boruta
    HAS_BORUTA = True
except Exception:
    HAS_BORUTA = False

try:
    import shap  # pip install shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print("BORUTA disponible:", HAS_BORUTA, " | SHAP disponible:", HAS_SHAP)

if HAS_BORUTA and 'Xm' in globals():
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
    boruta = BorutaPy(estimator=rf, n_estimators="auto", random_state=SEED, verbose=1, max_iter=50)
    boruta.fit(Xm.values, ym.values)
    cols_boruta = [c for c, keep in zip(Xm.columns, boruta.support_) if keep]
    print(f"BORUTA seleccionó {len(cols_boruta)} columnas")

if HAS_SHAP and 'model' in globals() and 'Xm' in globals():
    explainer = shap.TreeExplainer(model)
    smp = min(1000, len(Xm))
    shap_values = explainer.shap_values(Xm.sample(smp, random_state=SEED))
    print("SHAP values shape:", np.array(shap_values).shape)
